# Improved Hierarchical Multi-Label Classification with LLM Labels
## Uses pre-generated LLM labels (val_data, seed_train_labels) for better initial training

In [1]:
import os
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Path Configuration
ROOT = r"F:/work/BIGdata/project_release/Amazon_products"
TRAIN_PATH = os.path.join(ROOT, "train/train_corpus.txt")
TEST_PATH = os.path.join(ROOT, "test/test_corpus.txt")
HIERARCHY_PATH = os.path.join(ROOT, "class_hierarchy.txt")
KEYWORDS_PATH = os.path.join(ROOT, "class_related_keywords.txt")
MODEL_SAVE_DIR = r'F:/work/BIGdata/project_release/model'

# LLM generated data paths
LLM_DATA_PATH = os.path.join(ROOT, "llm_generated_data.pkl")  # val_data, seed_train_labels
FINAL_LABELS_PATH = os.path.join(MODEL_SAVE_DIR, "final_train_labels.pkl")  # final_train_labels

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# --- Data Loading ---
def load_data_and_graph():
    # Load training documents
    documents = []
    with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                documents.append({'id': int(parts[0]), 'text': parts[1]})
    
    # Load test documents
    test_documents = []
    if os.path.exists(TEST_PATH):
        with open(TEST_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t', 1)
                if len(parts) == 2:
                    test_documents.append({'id': int(parts[0]), 'text': parts[1]})
    
    # Load class information
    class_info = {}
    with open(KEYWORDS_PATH, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            parts = line.strip().split(':')
            if len(parts) == 2:
                name = parts[0].strip().replace('_', ' ')
                keywords = parts[1].strip()
                class_info[idx] = {'name': name, 'keywords': keywords}
    
    # Build hierarchy graph
    G = nx.DiGraph()
    with open(HIERARCHY_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            p, c = map(int, line.strip().split())
            G.add_edge(p, c)
    
    num_classes = len(class_info)
    roots = [n for n, d in G.in_degree() if d == 0]
    
    return documents, test_documents, G, class_info, num_classes, roots

all_docs, test_docs, G, class_info, num_classes, roots = load_data_and_graph()
print(f"Loaded {len(all_docs)} training docs, {len(test_docs)} test docs, and {num_classes} classes")

Loaded 29487 training docs, 19658 test docs, and 531 classes


In [3]:
# --- Load Pre-generated LLM Labels ---

def load_llm_labels():
    """
    Load pre-generated LLM labels from pickle files
    Priority: 
    1. llm_generated_data.pkl (val_data, seed_train_labels, unlabeled_indices)
    2. final_train_labels.pkl (final_train_labels)
    """
    val_data = None
    seed_train_labels = None
    unlabeled_indices = None
    final_train_labels = None
    
    # Try loading llm_generated_data.pkl first
    if os.path.exists(LLM_DATA_PATH):
        print(f"Loading LLM labels from {LLM_DATA_PATH}...")
        with open(LLM_DATA_PATH, 'rb') as f:
            data = pickle.load(f)
            val_data = data.get('val_data', [])
            seed_train_labels = data.get('seed_train_labels', {})
            unlabeled_indices = data.get('unlabeled_indices', [])
        
        print(f"  - Validation data: {len(val_data)} samples")
        print(f"  - Seed train labels: {len(seed_train_labels)} samples")
        print(f"  - Unlabeled indices: {len(unlabeled_indices)} samples")
    
    # Try loading final_train_labels.pkl
    if os.path.exists(FINAL_LABELS_PATH):
        print(f"\nLoading final train labels from {FINAL_LABELS_PATH}...")
        with open(FINAL_LABELS_PATH, 'rb') as f:
            final_train_labels = pickle.load(f)
        print(f"  - Final train labels: {len(final_train_labels)} samples")
    
    return val_data, seed_train_labels, unlabeled_indices, final_train_labels

val_data, seed_train_labels, unlabeled_indices, final_train_labels = load_llm_labels()

# Decide which labels to use
if final_train_labels is not None:
    print("\n==> Using final_train_labels for training")
    use_final_labels = True
elif seed_train_labels is not None:
    print("\n==> Using seed_train_labels for training")
    use_final_labels = False
else:
    raise ValueError("No pre-generated labels found!")

Loading LLM labels from F:/work/BIGdata/project_release/Amazon_products\llm_generated_data.pkl...
  - Validation data: 200 samples
  - Seed train labels: 600 samples
  - Unlabeled indices: 28687 samples

==> Using seed_train_labels for training


In [4]:
# --- Hierarchical Constraint Functions ---

def get_ancestors(G, node):
    """Get all ancestor nodes in the hierarchy"""
    ancestors = set()
    queue = [node]
    while queue:
        current = queue.pop(0)
        for parent in G.predecessors(current):
            if parent not in ancestors:
                ancestors.add(parent)
                queue.append(parent)
    return ancestors

def enforce_hierarchy_constraint(labels, G):
    """
    Enforce hierarchical consistency:
    If a node is labeled, all its ancestors must also be labeled
    """
    constrained_labels = set(labels)
    
    for label in labels:
        ancestors = get_ancestors(G, label)
        constrained_labels.update(ancestors)
    
    return list(constrained_labels)

# Apply hierarchy constraint to all labels
print("\nApplying hierarchy constraints to labels...")

if use_final_labels:
    # Apply to final_train_labels
    for doc_id in final_train_labels:
        original = final_train_labels[doc_id]
        constrained = enforce_hierarchy_constraint(original, G)
        final_train_labels[doc_id] = constrained
    print(f"Applied hierarchy constraint to {len(final_train_labels)} final labels")
else:
    # Apply to seed_train_labels
    for idx in seed_train_labels:
        original = seed_train_labels[idx]
        constrained = enforce_hierarchy_constraint(original, G)
        seed_train_labels[idx] = constrained
    print(f"Applied hierarchy constraint to {len(seed_train_labels)} seed labels")

# Apply to validation data
if val_data:
    for item in val_data:
        original = item['labels']
        constrained = enforce_hierarchy_constraint(original, G)
        item['labels'] = constrained
    print(f"Applied hierarchy constraint to {len(val_data)} validation labels")


Applying hierarchy constraints to labels...
Applied hierarchy constraint to 600 seed labels
Applied hierarchy constraint to 200 validation labels


In [5]:
# --- Silver Labeling for Unlabeled Data ---

def generate_silver_labels_tfidf(documents, unlabeled_indices, class_info, num_classes):
    """
    Generate silver labels for unlabeled documents using TF-IDF
    """
    print("\nGenerating silver labels for unlabeled data...")
    
    # Prepare class descriptions
    class_descriptions = []
    for i in range(num_classes):
        desc = f"{class_info[i]['name']} {class_info[i]['keywords']}"
        class_descriptions.append(desc)
    
    # Get unlabeled documents
    unlabeled_docs = [documents[i] for i in unlabeled_indices]
    doc_texts = [d['text'] for d in unlabeled_docs]
    
    # TF-IDF
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
    all_texts = doc_texts + class_descriptions
    vectorizer.fit(all_texts)
    
    doc_vectors = vectorizer.transform(doc_texts)
    class_vectors = vectorizer.transform(class_descriptions)
    
    # Compute similarity
    similarity = cosine_similarity(doc_vectors, class_vectors)
    
    # Generate labels
    silver_labels = {}
    for doc_idx, unlabeled_idx in enumerate(unlabeled_indices):
        # Get top-k similar classes
        top_k = 5
        top_classes = np.argsort(similarity[doc_idx])[-top_k:].tolist()
        
        # Adaptive threshold: top-k 중 perceptile 85% 이상만 남김 
        # perceptile: 
        threshold = np.percentile(similarity[doc_idx], 85)
        top_classes = [c for c in top_classes if similarity[doc_idx][c] > threshold]
        
        # Ensure 2-3 labels:
        if len(top_classes) < 2: # 라벨이 너무 적으면 2개로 늘림 
            top_classes = np.argsort(similarity[doc_idx])[-2:].tolist()
        elif len(top_classes) > 3: #라벨이 3개를 초과하면 줄임(사전 info로 label은 최대 3개까지만 가질 수 있음을 확인)
            top_classes = top_classes[-3:]
        
        # Apply hierarchy constraint
        top_classes = enforce_hierarchy_constraint(top_classes, G)
        
        # Limit to 3 most confident
        if len(top_classes) > 3:
            scores = [(c, similarity[doc_idx][c]) for c in top_classes]
            scores.sort(key=lambda x: x[1], reverse=True)
            top_classes = [c for c, _ in scores[:3]]
        
        silver_labels[unlabeled_idx] = top_classes
    
    return silver_labels

# Generate silver labels if needed
if not use_final_labels and unlabeled_indices:
    silver_labels = generate_silver_labels_tfidf(all_docs, unlabeled_indices, class_info, num_classes)
    print(f"Generated silver labels for {len(silver_labels)} unlabeled documents")
    
    # Combine seed and silver labels
    combined_train_labels = {**seed_train_labels, **silver_labels}
    print(f"Total training labels: {len(combined_train_labels)}")
else:
    combined_train_labels = final_train_labels
    print(f"Using {len(combined_train_labels)} final training labels")


Generating silver labels for unlabeled data...
Generated silver labels for 28687 unlabeled documents
Total training labels: 29287


In [6]:
# --- Build Adjacency Matrix ---

def build_adjacency_matrix(G, num_classes):
    A = torch.eye(num_classes)
    
    for u, v in G.edges():
        A[u, v] = 1
        A[v, u] = 1
    
    # Normalize
    degree = A.sum(dim=1)
    degree_inv_sqrt = torch.pow(degree, -0.5)
    degree_inv_sqrt[torch.isinf(degree_inv_sqrt)] = 0
    D_inv_sqrt = torch.diag(degree_inv_sqrt)
    
    A_hat = D_inv_sqrt @ A @ D_inv_sqrt
    
    return A_hat

A_hat = build_adjacency_matrix(G, num_classes).to(device)
print(f"Adjacency matrix shape: {A_hat.shape}")

Adjacency matrix shape: torch.Size([531, 531])


In [7]:
# --- Dataset Definition ---

class TextDataset(Dataset):
    def __init__(self, texts, labels_dict, tokenizer, num_classes, max_length=128):
        self.texts = texts
        self.labels_dict = labels_dict
        self.tokenizer = tokenizer
        self.num_classes = num_classes
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        label_vector = torch.zeros(self.num_classes)
        if idx in self.labels_dict:
            for label in self.labels_dict[idx]:
                if label < self.num_classes:
                    label_vector[label] = 1
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': label_vector
        }

In [8]:
# --- Model Definition ---

class ImprovedGCNClassifier(nn.Module):
    def __init__(self, num_classes, bert_model='bert-base-uncased', hidden_dim=768, gcn_layers=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model)
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        
        # GCN layers
        self.gcn_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(gcn_layers)
        ])
        
        # Class embeddings
        self.class_embeddings = nn.Parameter(torch.randn(num_classes, hidden_dim))
        
        # Attention
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
        
        self.layer_norm = nn.LayerNorm(hidden_dim)
    
    def forward(self, input_ids, attention_mask, A_hat):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        doc_embed = bert_output.last_hidden_state[:, 0, :]
        
        # GCN
        class_feats = self.class_embeddings
        for gcn_layer in self.gcn_layers:
            class_feats = F.relu(gcn_layer(A_hat @ class_feats))
            class_feats = self.layer_norm(class_feats)
        
        # Attention
        doc_expanded = doc_embed.unsqueeze(1)
        class_expanded = class_feats.unsqueeze(0).expand(doc_embed.size(0), -1, -1)
        
        attn_output, _ = self.attention(doc_expanded, class_expanded, class_expanded)
        attn_output = attn_output.squeeze(1)
        
        # Combine and classify
        combined = torch.cat([doc_embed, attn_output], dim=1)
        logits = self.classifier(combined)
        
        return logits

In [9]:
# --- Loss and Training Functions ---

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

def train_epoch(model, loader, optimizer, criterion, A_hat, device):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Training"):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(ids, mask, A_hat)
        loss = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, A_hat, device, threshold=0.5):
    model.eval()
    preds, targets = [], []
    
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            
            logits = model(ids, mask, A_hat)
            pred = (torch.sigmoid(logits) > threshold).float()
            
            preds.append(pred.cpu().numpy())
            targets.append(lbls.cpu().numpy())
    
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    
    # Micro F1
    tp = np.sum(preds * targets)
    fp = np.sum(preds * (1 - targets))
    fn = np.sum((1 - preds) * targets)
    micro_f1 = 2 * tp / (2 * tp + fp + fn + 1e-10)
    
    # Macro F1
    tp_per_class = np.sum(preds * targets, axis=0)
    fp_per_class = np.sum(preds * (1 - targets), axis=0)
    fn_per_class = np.sum((1 - preds) * targets, axis=0)
    f1_per_class = 2 * tp_per_class / (2 * tp_per_class + fp_per_class + fn_per_class + 1e-10)
    macro_f1 = np.mean(f1_per_class)
    
    return micro_f1, macro_f1

In [10]:
# --- Prepare Training Data ---

# Create doc_id to index mapping
id_to_doc = {d['id']: d for d in all_docs}

# Prepare training data from combined_train_labels
train_texts = []
train_labels_remapped = {}

# combined_train_labels can have doc_id or doc_index as keys
# Need to handle both cases
for key, labels in combined_train_labels.items():
    if isinstance(key, int):
        # Check if it's a doc_id or index
        if key in id_to_doc:
            # It's a doc_id
            doc = id_to_doc[key]
            train_texts.append(doc['text'])
            train_labels_remapped[len(train_texts) - 1] = labels
        elif 0 <= key < len(all_docs):
            # It's an index
            doc = all_docs[key]
            train_texts.append(doc['text'])
            train_labels_remapped[len(train_texts) - 1] = labels

print(f"Prepared {len(train_texts)} training samples")

# Prepare validation data
if val_data:
    val_texts = [item['text'] for item in val_data]
    val_labels_remapped = {i: item['labels'] for i, item in enumerate(val_data)}
    print(f"Prepared {len(val_texts)} validation samples")
else:
    # If no LLM validation data, use 20% of training for validation
    from sklearn.model_selection import train_test_split
    train_indices = list(range(len(train_texts)))
    train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42)
    
    val_texts = [train_texts[i] for i in val_idx]
    val_labels_remapped = {i: train_labels_remapped[val_idx[i]] for i in range(len(val_idx))}
    
    train_texts = [train_texts[i] for i in train_idx]
    new_train_labels = {i: train_labels_remapped[train_idx[i]] for i in range(len(train_idx))}
    train_labels_remapped = new_train_labels
    
    print(f"Split into {len(train_texts)} train and {len(val_texts)} validation samples")

Prepared 28687 training samples
Prepared 200 validation samples


In [ ]:
# --- Main Training Loop ---

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

train_ds = TextDataset(train_texts, train_labels_remapped, tokenizer, num_classes)
val_ds = TextDataset(val_texts, val_labels_remapped, tokenizer, num_classes)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

model = ImprovedGCNClassifier(num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = FocalLoss(alpha=0.25, gamma=2.0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

best_f1 = 0
SELF_TRAIN_ROUNDS = 3
EPOCHS_PER_ROUND = 5

print("\n=== Starting Training ===")

for round_num in range(SELF_TRAIN_ROUNDS):
    print(f"\n{'='*50}")
    print(f"Round {round_num + 1}/{SELF_TRAIN_ROUNDS}")
    print(f"{'='*50}")
    
    confidence_threshold = 0.75 - (round_num * 0.1)
    print(f"Confidence threshold: {confidence_threshold:.2f}")
    
    for epoch in range(EPOCHS_PER_ROUND):
        loss = train_epoch(model, train_loader, optimizer, criterion, A_hat, device)
        micro_f1, macro_f1 = evaluate(model, val_loader, A_hat, device)
        scheduler.step()
        
        print(f"  Epoch {epoch+1}/{EPOCHS_PER_ROUND}: Loss={loss:.4f}, "
              f"Micro-F1={micro_f1:.4f}, Macro-F1={macro_f1:.4f}")
        
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            model_path = os.path.join(MODEL_SAVE_DIR, "best_model_with_llm.pt")
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'f1': best_f1,
                'round': round_num,
                'epoch': epoch
            }, model_path)
            print(f"    -> Best model saved! F1={best_f1:.4f}")
    
    # Pseudo-label update
    if round_num < SELF_TRAIN_ROUNDS - 1:
        print("\n  Updating pseudo-labels...")
        model.eval()
        
        inference_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
        new_labels = {}
        
        with torch.no_grad():
            batch_idx = 0
            for batch in tqdm(inference_loader, desc="Inference"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask, A_hat)
                probs = torch.sigmoid(logits).cpu()
                
                for i, prob_vec in enumerate(probs):
                    global_idx = batch_idx + i
                    
                    high_conf = (prob_vec > confidence_threshold).nonzero(as_tuple=True)[0].tolist()
                    
                    if len(high_conf) >= 2:
                        high_conf = enforce_hierarchy_constraint(high_conf, G)
                        if len(high_conf) > 3:
                            top_probs = [(c, prob_vec[c].item()) for c in high_conf]
                            top_probs.sort(key=lambda x: x[1], reverse=True)
                            high_conf = [c for c, _ in top_probs[:3]]
                        new_labels[global_idx] = high_conf
                    else:
                        new_labels[global_idx] = train_labels_remapped.get(global_idx, [])
                
                batch_idx += len(prob_vec)
        
        changed = sum(1 for k in new_labels if new_labels[k] != train_labels_remapped.get(k, []))
        print(f"  Updated {changed}/{len(new_labels)} labels")
        
        train_labels_remapped = new_labels
        train_ds = TextDataset(train_texts, train_labels_remapped, tokenizer, num_classes)
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

print(f"\n=== Training Complete. Best Macro F1: {best_f1:.4f} ===")

In [ ]:
# --- Test Prediction ---

def predict_test(model, test_docs, tokenizer, A_hat, device, G, batch_size=32):
    model.eval()
    
    dummy_labels = {i: [] for i in range(len(test_docs))}
    test_texts = [d['text'] for d in test_docs]
    
    test_ds = TextDataset(test_texts, dummy_labels, tokenizer, num_classes)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            logits = model(ids, mask, A_hat)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            for prob_vec in probs:
                top_indices = np.argsort(prob_vec)[-5:][::-1]
                predictions = [idx for idx in top_indices if prob_vec[idx] > 0.3]
                
                if len(predictions) < 2:
                    predictions = top_indices[:2].tolist()
                elif len(predictions) > 3:
                    predictions = predictions[:3]
                
                predictions = enforce_hierarchy_constraint(predictions, G)
                
                if len(predictions) > 3:
                    top_probs = [(p, prob_vec[p]) for p in predictions]
                    top_probs.sort(key=lambda x: x[1], reverse=True)
                    predictions = [p for p, _ in top_probs[:3]]
                
                all_predictions.append(predictions)
    
    return all_predictions

# Load and predict
checkpoint = torch.load(os.path.join(MODEL_SAVE_DIR, "best_model_with_llm.pt"))
model.load_state_dict(checkpoint['model_state_dict'])

predictions = predict_test(model, test_docs, tokenizer, A_hat, device, G)

# Save predictions
output_path = os.path.join(MODEL_SAVE_DIR, "predictions_with_llm.txt")
with open(output_path, 'w') as f:
    for doc, preds in zip(test_docs, predictions):
        pred_str = ' '.join(map(str, sorted(preds)))
        f.write(f"{doc['id']}\t{pred_str}\n")

print(f"\nPredictions saved to {output_path}")
print(f"Avg predictions per document: {np.mean([len(p) for p in predictions]):.2f}")